# ⚡ Tự Động Hóa Toàn Diện Backtest Trên Google Colab Pro (High-RAM)
### Chiến Lược: `NostalgiaForInfinityX8_Combined_AI` + AI Gatekeeper v3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tminhnhat/NostalgiaForInfinity/blob/main/notebooks/Run_NFIX8_Backtest_Colab_HighRAM.ipynb)

---
### 🤖 QUY TRÌNH TỰ ĐỘNG HÓA 100% (ZERO-TOUCH RUN):
1. **Cấu hình Runtime ban đầu (chỉ chọn 1 lần):**
   - Vào **Runtime (Thời gian chạy)** ➔ **Change runtime type (Thay đổi loại thời gian chạy)**.
   - **Hardware accelerator**: Chọn **None** (CPU - Tiết kiệm Compute Units tối đa).
   - **Runtime shape**: Chọn **High-RAM** (Cấp 8 vCPUs & ~52GB RAM).
2. **Thao tác duy nhất:** Bấm tổ hợp phím **`Ctrl + F9`** (hoặc chọn menu **Runtime ➔ Run all**).
3. **Hệ thống tự động thực hiện từ A - Z:**
   - ⚙️ Tự động cài đặt TA-Lib C-library, Freqtrade & LightGBM.
   - 🔄 Tự động kéo mã nguồn mới nhất từ GitHub.
   - 📥 Tự động tải nến Binance Vision (không bị giới hạn IP Mỹ 451).
   - 🚀 Tự động chạy Backtest 185 ngày trên 8 luồng song song (~3 - 5 phút).
   - 📊 Tự động hiển thị báo cáo hiệu suất chi tiết (Winrate, Net Profit, DD, Sharpe).
   - 📱 **Tự động gửi thông báo tóm tắt & file báo cáo về Telegram cá nhân**.
   - 🛑 **Tự động ngắt kết nối Runtime (`runtime.unassign()`)** để dừng tính tiền Compute Units ngay lập tức!

In [ ]:
# ==============================================================================
# BƯỚC 1: KIỂM TRA PHẦN CỨNG HỆ THỐNG & TÀI NGUYÊN COLAB PRO
# ==============================================================================
import os
import psutil

vcpus = os.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
disk_gb = psutil.disk_usage('/').total / (1024 ** 3)
disk_free_gb = psutil.disk_usage('/').free / (1024 ** 3)

print(f"🖥️  Số luồng CPU (vCPUs): {vcpus} cores")
print(f"🧠  Dung lượng RAM:       {ram_gb:.1f} GB")
print(f"💾  Ổ đĩa lưu trữ:       {disk_free_gb:.1f} GB trống / {disk_gb:.1f} GB")

if ram_gb >= 30:
    print("\n✅ ĐÃ XÁC NHẬN: Môi trường High-RAM (52GB RAM, 8 vCPUs) đang kích hoạt!")
else:
    print("\n⚠️  LƯU Ý: Đang ở Standard RAM (~12GB). Khuyến nghị vào Runtime -> Change runtime type -> Chọn High-RAM để đạt tốc độ 3-5 phút!")

In [ ]:
# ==============================================================================
# BƯỚC 2: CÀI ĐẶT TA-LIB C-LIBRARY VÀ FREQTRADE (SIÊU TỐC ~1-2 PHÚT)
# ==============================================================================
import sys
import subprocess

print("⏳ Đang kiểm tra và cài đặt thư viện kỹ thuật TA-Lib C...")
install_talib_script = """
if [ ! -f /usr/lib/libta_lib.so.0 ]; then
    wget -q http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
    tar -xzf ta-lib-0.4.0-src.tar.gz
    cd ta-lib/
    ./configure --prefix=/usr > /dev/null 2>&1
    make -j$(nproc) > /dev/null 2>&1
    sudo make install > /dev/null 2>&1
    sudo ldconfig > /dev/null 2>&1
    cd ..
    rm -rf ta-lib ta-lib-0.4.0-src.tar.gz
    echo '✅ Đã cài đặt xong TA-Lib C library!'
else
    echo '✅ TA-Lib C library đã sẵn sàng!'
fi
"""
subprocess.run(install_talib_script, shell=True, check=True, executable='/bin/bash')

print("⏳ Đang cài đặt Freqtrade, LightGBM và các gói phụ thuộc...")
!pip install -q ta-lib freqtrade lightgbm python-rapidjson pyarrow fastparquet

# Xác minh cài đặt thành công
import talib
import freqtrade
import lightgbm
import rapidjson
print(f"\n🎉 XÁC NHẬN SẴN SÀNG:")
print(f"   - Freqtrade: {freqtrade.__version__}")
print(f"   - TA-Lib:    {talib.__version__}")
print(f"   - LightGBM:  {lightgbm.__version__}")

In [ ]:
# ==============================================================================
# BƯỚC 3: CLONE HOẶC CẬP NHẬT MÃ NGUỒN BOT MỚI NHẤT TỪ GITHUB
# ==============================================================================
import os
from pathlib import Path

REPO_URL = "https://github.com/tminhnhat/NostalgiaForInfinity.git"
REPO_DIR = "/content/NostalgiaForInfinity"

if not os.path.exists(REPO_DIR):
    print(f"📥 Đang clone repository từ: {REPO_URL}...")
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"🔄 Đang cập nhật repository mới nhất...")
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}

# Kiểm tra xác nhận file chiến lược & mô hình AI
assert Path("NostalgiaForInfinityX8_Combined_AI.py").exists(), "Thiếu file chiến lược!"
assert Path("configs/backtest_config.json").exists(), "Thiếu file config backtest!"
assert Path("user_data/models/nfix8_short_ai_v3.pkl").exists(), "Thiếu file mô hình AI Gatekeeper v3!"

print("\n✅ Thư mục làm việc hiện tại:", os.getcwd())
print("✅ Chiến lược & Mô hình AI Gatekeeper v3 đã sẵn sàng!")

In [ ]:
# ==============================================================================
# BƯỚC 4: TẢI NẾN BINANCE VISION & ÁP DỤNG PATCH BYPASS GEO-RESTRICTION
# ==============================================================================
import os
from pathlib import Path

print("📥 Đang tải nến 185 ngày từ Binance Vision (Không dùng API sàn, không bị chặn IP Mỹ 451)...")
!python scripts/download_candles.py \
    --config configs/backtest_config.json \
    --days 185 \
    --output-dir user_data/data/binance \
    --workers 24 \
    --skip-existing

print("\n🛡️ Áp dụng Offline Markets Patch...")
!python scripts/patch_offline_markets.py

print("\n🎉 Dữ liệu nến đã sẵn sàng trong thư mục user_data/data/binance/futures!")

In [ ]:
# ==============================================================================
# BƯỚC 5: CHẠY BACKTEST ĐÁNH GIÁ SIÊU TỐC (8 THREADS, ~3 - 5 PHÚT)
# ==============================================================================
import time

start_time = time.time()
print("🚀 BẮT ĐẦU CHẠY BACKTEST TRÊN 8 vCPUs (HIGH-RAM)...")

!python scripts/run_eval_backtest.py \
    --strategy NostalgiaForInfinityX8_Combined_AI \
    --days 185 \
    --run-label "Colab_Pro_Auto_Run"

elapsed = time.time() - start_time
print(f"\n⏱️ TỔNG THỜI GIAN THỰC THI: {elapsed / 60:.2f} phút ({elapsed:.1f} giây)!")

In [ ]:
# ==============================================================================
# BƯỚC 6: HIỂN THỊ BÁO CÁO HIỆU SUẤT TRỰC QUAN (MARKDOWN FORMATTED)
# ==============================================================================
from IPython.display import display, Markdown
from pathlib import Path

report_file = Path("reports/latest_report.md")
if report_file.exists():
    with open(report_file, "r", encoding="utf-8") as f:
        content = f.read()
    display(Markdown(content))
else:
    print("⚠️ Không tìm thấy file reports/latest_report.md!")

In [ ]:
# ==============================================================================
# BƯỚC 7: TỰ ĐỘNG GỬI BÁO CÁO TÓM TẮT & FILE KẾT QUẢ QUA TELEGRAM
# ==============================================================================
!python scripts/notify_telegram.py \
    --report reports/latest_report.md \
    --context-title "⚡ BÁO CÁO BACKTEST COLAB PRO (HIGH-RAM)"

In [ ]:
# ==============================================================================
# BƯỚC 8: TỰ ĐỘNG NGẮT KẾT NỐI RUNTIME ĐỂ TIẾT KIỆM TÍN DỤNG COMPUTE UNITS
# ==============================================================================
# Nếu bật True: Sau khi backtest và gửi Telegram xong, Colab sẽ tự ngắt máy
# giúp anh không bao giờ bị lãng phí tín dụng nếu quên tắt tab trình duyệt.
AUTO_DISCONNECT = True

if AUTO_DISCONNECT:
    print("🛑 Đang hoàn tất và tự động ngắt kết nối Google Colab Pro...")
    try:
        from google.colab import runtime
        runtime.unassign()
        print("✅ Đã ngắt kết nối runtime an toàn!")
    except Exception as e:
        print(f"⚠️ Không thể ngắt runtime tự động: {e}")
else:
    print("ℹ️ AUTO_DISCONNECT đang tắt. Runtime vẫn giữ kết nối để anh tiếp tục làm việc.")